# EDA - Mortalidad Buenos Aires
01_exploratory_analysis

## Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

# Uso general
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from etl.extract.load_data import load_defunciones
from etl.transform.cleaning import (
    drop_muerte_materna,
    normalizar_cie10_upper,
    mapear_cie10_faltantes,
    eliminar_jurisdiccion_nula,
    eliminar_sexo_desconocido,
    eliminar_edad_sin_especificar,
    limpiar_etiqueta_grupo_edad,
    filtrar_buenos_aires,
)
from etl.transform.feature_engineering import (
    clasificar_cie10,
    transformacion_manual,
    transformacion_pipeline,
)

## Lectura DataFrame

In [ ]:
df1 = load_defunciones('../data/raw/defunciones-ocurridas-y-registradas-en-la-republica-argentina-entre-los-anos-2005-2022.csv')

In [ ]:
df1.shape

In [ ]:
df1.head(5)

## Limpieza de datos
Variables :
* muerte_materna_id
* muerte_materna_clasificacion

In [ ]:
df1.isna().sum()

Descubrimos que hay espacios "", que no son considerados nulos en la columna "muerte_materna_id", tambien sumado a esto , se encontro que mas del 90% de los datos en las columnas "muerte_materna_id", "muerte_materna_clasificacion" son nulos, esto implica que las 2 son columnas casi vacias sin informacion que podamos recuperar y se opto que la mejor opcion es eliminarlas ya que su informacion es practicamente nula y no se relacionan con otras columnas como para intentar su recuperacion

In [ ]:
df = drop_muerte_materna(df1)

## Limpieza de datos
Variable :
* cie10_causa_id

In [ ]:
df.isna().sum()

In [ ]:
causas_unicas = df.groupby('cie10_causa_id')['cie10_clasificacion'].nunique()

# 2. Filtramos para ver si algún código tiene MÁS de una descripción asociada
no_unicos = causas_unicas[causas_unicas > 1]

print(no_unicos.size)

como podemos observar, dentro de estos casos nulos hay errores de mayusculas y minusculas
<br>Por lo tanto

<br>ARREGLAMOS LOWER/UPPER CASE

In [ ]:
df = normalizar_cie10_upper(df)
causas_unicas = df.groupby('cie10_causa_id')['cie10_clasificacion'].nunique()
nulos = causas_unicas[causas_unicas == 0]
print(nulos)

Vemos que pese a nuestro arreglo, aun siguen habiendo causas_ids que no tienen un cie10_clasificacion, buscando el significado de los codigos ci-10 de la OMS y teniendo en contexto la situacion de pandemia durante el periodo del dataset.
decidimos manualmente rellenar los datos.

In [ ]:
df = mapear_cie10_faltantes(df)

In [ ]:
df.isna().sum()

## Limpieza de datos
Variable :
* jurisdicion_residencia_nombre

In [ ]:
df['jurisdiccion_de_residencia_id'].unique()

In [ ]:
df.groupby('jurisdiccion_de_residencia_id')['jurisdicion_residencia_nombre'].unique()

In [ ]:
df = eliminar_jurisdiccion_nula(df)

In [ ]:
df.isna().sum()

## Limpieza de datos
Validacion de seguridad (Busqueda de "") que no se detecten como nulos

In [ ]:
# Hace un mapeo en el dataframe
# Lambda funciona como funcion de una linea, busca
# donde x, transformado a str, quitando los espacios en blanco, es igual a un espacio vacio puro
filtro = df.map(lambda x: str(x).strip() == "")

# Filtrar el DataFrame para ver solo las filas que contienen esa palabra
# Recordar, axis=0 = columna
# axis=1 = fila
# any = indica "algun", en este caso , alguna fila
resultados = df[filtro.any(axis=1)]
resultados.head(2)

Luego de comprobar que no hay valores nulos ni espacios vacios como encontramos que ocurrio antes, se da el dataSet por limpio

## Limpieza de datos
Filtros y limpiezas para trabajar el DataSet

Limpieza de "Desconocido" en Genero

In [ ]:
df = eliminar_sexo_desconocido(df)
df.shape

Limpieza de rango de edad

In [ ]:
df['grupo_edad'].unique()

In [ ]:
df = eliminar_edad_sin_especificar(df)
df.shape

Modificando Rango edad

In [ ]:
df = limpiar_etiqueta_grupo_edad(df)

Filtro de residencia

In [ ]:
df['jurisdicion_residencia_nombre'].value_counts()

In [ ]:
df = filtrar_buenos_aires(df)
df.shape

## Feature Engineering

Pasamos las cie10_clasificacion a una variable mas general para agrupar las categorias y trabajar mejos los datos

## Transformacion Manual

In [ ]:
df_manual = transformacion_manual(df)
df_manual.head(5)

In [ ]:
df_manual['grupo_edad'].unique()

## Transformacion con Pipeline

In [ ]:
X_transf, pipeline = transformacion_pipeline(df)
print(X_transf)

## Análisis de correlación

In [ ]:
# Analisis rapido de correlacion con matriz de correlacion

plt.figure(figsize=(10, 8))

# Eliminamos las variables cualitativas
df_corre = df_manual.copy()

matriz_corr = df_corre.corr(numeric_only=True)

# Creamos el mapa de calor
sns.heatmap(
    matriz_corr,
    annot=True,          # annot=True dibuja los números dentro de cada celda
    cmap='coolwarm',     # Escala de color: azul para negativo, rojo para positivo
    fmt=".2f",           # Muestra solo 2 decimales
    linewidths=0.5,      # Deja una pequeña línea de separación entre celdas
    vmin=-1, vmax=1      # Fuerza a que la escala de color vaya estricta de -1 a 1
)

plt.title('Matriz de Correlación de Variables')
plt.show()

No existe una relacion lineal, asi que tenemos que analizar mas a fondo

In [ ]:
causas = df['cie10_clasificacion'].unique()
len(causas)

In [ ]:
causas = df['cie10_clasificacion'].value_counts()
total = causas.head(3)
total.values.sum()

In [ ]:
sorted(df['anio'].unique())

RECORDAR : HAY UNA COLUMNA "CANTIDAD", QUE INDICA LA CANTIDAD DE MUERTOS

In [ ]:
df['supracategoria'] = df['cie10_causa_id'].apply(clasificar_cie10)
df.head(3)

In [ ]:
dfGraf = df.copy()

cant = dfGraf.groupby(['supracategoria'])['cantidad'].sum().reset_index(name='cantidad_casos')
topCant = cant['cantidad_casos'].sort_values(ascending=False).head(3)
filt = cant[cant['cantidad_casos'].isin(topCant.values)]
clasi = filt['supracategoria'].values
clasi

In [ ]:
# Grafico para ver relacion de cantidad, con año y separar por causa de defuncion
dfFilt = dfGraf[dfGraf['supracategoria'].isin(clasi)]

df_agrupado = dfFilt.groupby(['anio', 'supracategoria'])['cantidad'].sum().reset_index(name='cantidad_casos')

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_agrupado,
    x='anio',
    y='cantidad_casos',
    marker='o',
    hue='supracategoria'
)
anios_reales = sorted(df_agrupado['anio'].unique())

plt.xticks(ticks=anios_reales, labels=[int(a) for a in anios_reales])

plt.title('Top 3 defunciones anuales de Buenos Aires(2005-2022) ')
plt.ylabel('Total de defunciones(Fallecidos)')
plt.xlabel('Años')
plt.show()

In [ ]:
# Grafico para ver relacion entre cantidad, con rando edad y separar por causa de defuncion
dfFilt = dfGraf[dfGraf['supracategoria'].isin(clasi)]

df_agrupado = dfFilt.groupby(['grupo_edad', 'supracategoria'])['cantidad'].sum().reset_index(name='cantidad_casos')

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_agrupado,
    x='grupo_edad',
    y='cantidad_casos',
    marker='o',
    hue='supracategoria'
)
ordEdad = ['De a 0 a 14 anios', 'De 15 a 34 anios',
          'De 35 a 54 anios', 'De 55 a 74 anios',
          'De 75 anios y mas']

plt.xticks(ticks=[0, 1, 2, 3, 4], labels=[a for a in ordEdad])

plt.title('Top 3 defunciones por rango de edad de Buenos Aires(2005-2022) ')
plt.ylabel('Total de defunciones(Fallecidos)')
plt.xlabel('Edad')
plt.show()